# 3 · Ejercicio 2: isomorfismos (15 min)

Dos tipos son **isomorfos** si puedes ir de uno a otro (`from`) y volver (`to`) sin perder nada. Te piden:
- **a)** demostrarlo con aritmética (contando valores),
- **b)** implementar `from` y `to`.

### Preámbulo
Ejecuta esta celda antes que las demás: son las mismas definiciones que trae el examen.

In [ ]:
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

trait Isomorphic[A, B]:

    def from(a: A): B

    def to(b: B): A

    // equality

    def equalA(a1: A, a2: A): Boolean =
        a1 == a2

    def equalB(b1: B, b2: B): Boolean =
        b1 == b2

    // Bijection laws

    def law1(a: A): Boolean =
        equalA(to(from(a)), a)

    def law2(b: B): Boolean =
        equalB(from(to(b)), b)

// type Tree[A] = 1 + Tree[A] * A * Tree[A]

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:

    def void[A]: Tree[A] =
        Empty()

    def leaf[A](a: A): Node[A] =
        Node(Empty(), a, Empty())

    def right[A](a: A, tree: Tree[A]): Node[A] =
        Node(Empty(), a, tree)

    def left[A](tree: Tree[A], a: A): Node[A] =
        Node(tree, a, Empty())

    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] =
        Node(left, a, right)

import Tree._

def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
    tree match
        case Empty() => empty
        case Node(left, a, right) =>
            node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

## a) Contar valores

| Tipo | Cardinalidad |
|---|---|
| `Nothing` / `Unit` / `Boolean` | 0 / 1 / 2 |
| `Either[A, B]` | \|A\| **+** \|B\| |
| `(A, B)` | \|A\| **·** \|B\| |
| `Option[A]` | \|A\| + 1 |
| `A => B` | \|B\| **^** \|A\| (¡base = resultado!) |

Se escribe una igualdad por línea aplicando las reglas del álgebra:

```
|Option[A] => B| =
|B| ^ |Option[A]| =
|B| ^ (|A| + 1) =
|B| ^ |A| * |B| ^ 1 =
|A => B| * |B| =
|(A => B, B)|
```

Reglas que más salen: `a·(b+c) = ab+ac`, `a^(b+c) = a^b·a^c`, `(a·b)^c = a^c·b^c`, `a^1 = a`, conmutativa.

Si te dicen "**demuestra o refuta**" y las cuentas no salen iguales, no son isomorfos: lo justificas con un ejemplo (por ejemplo `A = Unit`).

## b) La receta de `from` / `to`

1. Haz una **tabla**: cada forma del tipo izquierdo ↔ su forma en el derecho.
2. `from` = la tabla de izquierda a derecha, con `match`. `to` = de derecha a izquierda.
3. Si la entrada es una **función** con dominio pequeño, evalúala en todos los valores: `f(false)`, `f(true)`; `f(None)`, `a => f(Some(a))`; `b => f(Left(b))`, `c => f(Right(c))`.
4. Si tienes que **devolver una función**, usa una lambda o un bloque de `case`.

## Ejemplo 1 · `Option[A] => B` ≅ `(B, A => B)`

In [ ]:
class Iso1[A, B] extends Isomorphic[Option[A] => B, (B, A => B)]:

    def from(f: Option[A] => B): (B, A => B) =
        (f(None), a => f(Some(a)))

    def to(p: (B, A => B)): Option[A] => B =
        case None    => p._1
        case Some(a) => p._2(a)

## Ejemplo 2 · `(A, Either[B, C])` ≅ `Either[(A, B), (A, C)]`

Tabla: `(a, Left(b))` ↔ `Left((a, b))`; `(a, Right(c))` ↔ `Right((a, c))`.

In [ ]:
class Iso2[A, B, C] extends Isomorphic[(A, Either[B, C]), Either[(A, B), (A, C)]]:

    def from(t: (A, Either[B, C])): Either[(A, B), (A, C)] =
        t match
            case (a, Left(b))  => Left((a, b))
            case (a, Right(c)) => Right((a, c))

    def to(e: Either[(A, B), (A, C)]): (A, Either[B, C]) =
        e match
            case Left((a, b))  => (a, Left(b))
            case Right((a, c)) => (a, Right(c))

In [ ]:
val iso = Iso2[Int, Char, String]()
(iso.law1((1, Left('a'))), iso.law2(Right((2, "hola"))))

## Mini-ejercicio (5 min)

`Either[B, C] => A` ≅ `(B => A, C => A)`. Primero la cadena, luego el código.

In [ ]:
class IsoEj[A, B, C] extends Isomorphic[Either[B, C] => A, (B => A, C => A)]:

    def from(f: Either[B, C] => A): (B => A, C => A) =
        ???

    def to(t: (B => A, C => A)): Either[B, C] => A =
        ???

<details>
<summary><b>Ver solución</b></summary>

```
|Either[B, C] => A| = |A| ^ (|B| + |C|) = |A|^|B| * |A|^|C| = |B => A| * |C => A| = |(B => A, C => A)|
```

```scala
class IsoEj[A, B, C] extends Isomorphic[Either[B, C] => A, (B => A, C => A)]:

    def from(f: Either[B, C] => A): (B => A, C => A) =
        (b => f(Left(b)), c => f(Right(c)))

    def to(t: (B => A, C => A)): Either[B, C] => A =
        case Left(b)  => t._1(b)
        case Right(c) => t._2(c)
```

</details>

**Siguiente:** `04-ejercicio3-listas.ipynb`